In [ ]:
import os
import pandas as pd

# 定义基准目录路径
benchmark_dir = "./resources/benchmark"

# 获取所有子文件夹（即模型名称）
model_folders = [f for f in os.listdir(benchmark_dir) if os.path.isdir(os.path.join(benchmark_dir, f))]

print("检测到的模型文件夹：")
for folder in sorted(model_folders):
    print(f" - {folder}")

# 读取每个模型的 metrics.csv 文件，并收集所有类别（列名）
all_columns = set()

for model in model_folders:
    csv_path = os.path.join(benchmark_dir, model, "metrics.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        all_columns.update(df.columns.tolist())
    else:
        print(f"警告: {csv_path} 不存在")

print("\n所有 metrics.csv 中出现的类别（列名）：")
for col in sorted(all_columns):
    print(f" - {col}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === 路径设置 ===
benchmark_dir = "./resources/benchmark"
vis_dir = os.path.join(benchmark_dir, "visualization")
os.makedirs(vis_dir, exist_ok=True)

# === 模型顺序与简短名称映射 ===
model_order = [
    "baseline U-Net",
    "Model 1",
    "Model 2",
    "U-Net Multi Branch (permutation)",
    "U-Net Multi Branch (out conv)"
]

short_names = {
    "baseline U-Net": "Baseline",
    "Model 1": "M1",
    "Model 2": "M2",
    "U-Net Multi Branch (permutation)": "MB-Perm",
    "U-Net Multi Branch (out conv)": "MB-Out"
}

# 过滤存在的模型（以防某些缺失）
existing_models = [m for m in model_order if os.path.isdir(os.path.join(benchmark_dir, m))]
short_labels = [short_names[m] for m in existing_models]

# === 读取所有模型数据 ===
summary_data = []
for model in existing_models:
    csv_path = os.path.join(benchmark_dir, model, "metrics.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if len(df) == 1:
            row = df.iloc[0].to_dict()
            row["model"] = model
            summary_data.append(row)
        else:
            print(f"警告: {model} 的 metrics.csv 不是单行数据。")
    else:
        print(f"警告: {model} 的 metrics.csv 不存在。")

summary_df = pd.DataFrame(summary_data)
summary_df.set_index("model", inplace=True)
# 按指定顺序重排
summary_df = summary_df.reindex(existing_models)

# 保存汇总 CSV
summary_df.to_csv(os.path.join(vis_dir, "summary.csv"))
print("汇总 CSV 已保存。")

# === 数值列提取 ===
numeric_cols = summary_df.select_dtypes(include=[np.number]).columns.tolist()

# === 关键全局指标 ===
key_metrics = [
    "mIoU",
    "pixel_accuracy",
    "macro_f1_score",
    "fps",
    "max_gpu_memory_mb",
    "time_per_frame_sec"
]
key_metrics = [m for m in key_metrics if m in numeric_cols]

# === 绘图通用设置 ===
plt.rcParams.update({'font.size': 10})
x_pos = np.arange(len(short_labels))

def save_bar_and_line(metric, values, xlabel="Model", ylabel=None, title_prefix=""):
    """同时保存柱状图和折线图"""
    # --- 柱状图 ---
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(x_pos, values, color='steelblue', edgecolor='black', width=0.6)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}' if abs(h) < 1e4 else f'{h:.0f}',
                    xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(short_labels)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(f"{title_prefix}{metric}")
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"{metric}_bar.png"), dpi=200)
    plt.close(fig)

    # --- 折线图 ---
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(x_pos, values, marker='o', linewidth=2, markersize=6, color='darkorange')
    for i, v in enumerate(values):
        ax.annotate(f'{v:.3f}' if abs(v) < 1e4 else f'{v:.0f}',
                    xy=(i, v), textcoords="offset points",
                    xytext=(0, 8), ha='center', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(short_labels)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(f"{title_prefix}{metric}")
    ax.grid(True, linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"{metric}_line.png"), dpi=200)
    plt.close(fig)

# === 绘制全局指标 ===
for metric in key_metrics:
    values = summary_df[metric].values
    save_bar_and_line(metric, values)

# === Per-Class 可视化 ===
n_classes = 5
iou_cols = [f"class_{i}_iou" for i in range(n_classes)]
acc_cols = [f"class_{i}_accuracy" for i in range(n_classes)]
f1_cols = [f"class_{i}_f1" for i in range(n_classes)]

# 检查是否存在这些列
has_iou = all(col in numeric_cols for col in iou_cols)
has_acc = all(col in numeric_cols for col in acc_cols)
has_f1 = all(col in numeric_cols for col in f1_cols)

# 如果有 IoU，优先用 IoU；否则尝试 accuracy 或 F1
if has_iou:
    class_metrics = [(f"class_{i}_iou", f"Class {i} IoU") for i in range(n_classes)]
    metric_name_base = "iou"
elif has_acc:
    class_metrics = [(f"class_{i}_accuracy", f"Class {i} Acc") for i in range(n_classes)]
    metric_name_base = "accuracy"
elif has_f1:
    class_metrics = [(f"class_{i}_f1", f"Class {i} F1") for i in range(n_classes)]
    metric_name_base = "f1"
else:
    class_metrics = []
    metric_name_base = None

# --- (A) 每个模型 → 各类别（横向：类别，分组：模型）---
if class_metrics:
    n_models = len(short_labels)
    x_classes = np.arange(n_classes)
    total_width = 0.8
    bar_width = total_width / n_models

    # 柱状图：每个类别下，不同模型的值
    fig, ax = plt.subplots(figsize=(9, 5))
    for idx, model in enumerate(existing_models):
        values = [summary_df.loc[model, col] for col, _ in class_metrics]
        ax.bar(x_classes + idx * bar_width - total_width/2 + bar_width/2,
               values, bar_width, label=short_labels[idx])
    ax.set_xlabel("Class")
    ax.set_ylabel(f"{metric_name_base.upper()}")
    ax.set_title(f"Per-Class {metric_name_base.upper()} by Model")
    ax.set_xticks(x_classes)
    ax.set_xticklabels([f"C{i}" for i in range(n_classes)])
    ax.legend(title="Model")
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"per_class_{metric_name_base}_by_model_bar.png"), dpi=200)
    plt.close(fig)

    # 折线图：每个模型一条线，X=类别
    fig, ax = plt.subplots(figsize=(9, 5))
    for idx, model in enumerate(existing_models):
        values = [summary_df.loc[model, col] for col, _ in class_metrics]
        ax.plot(x_classes, values, marker='o', label=short_labels[idx], linewidth=2, markersize=6)
    ax.set_xlabel("Class")
    ax.set_ylabel(f"{metric_name_base.upper()}")
    ax.set_title(f"Per-Class {metric_name_base.upper()} by Model")
    ax.set_xticks(x_classes)
    ax.set_xticklabels([f"C{i}" for i in range(n_classes)])
    ax.legend(title="Model")
    ax.grid(True, linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"per_class_{metric_name_base}_by_model_line.png"), dpi=200)
    plt.close(fig)

# --- (B) 每个类别 → 各模型（横向：模型，分组：类别）---
if class_metrics:
    n_classes = len(class_metrics)
    x_models = np.arange(len(short_labels))
    total_width = 0.8
    bar_width = total_width / n_classes

    # 柱状图：每个模型下，各类别的值
    fig, ax = plt.subplots(figsize=(9, 5))
    for idx, (col, label) in enumerate(class_metrics):
        values = summary_df[col].values
        ax.bar(x_models + idx * bar_width - total_width/2 + bar_width/2,
               values, bar_width, label=label)
    ax.set_xlabel("Model")
    ax.set_ylabel(f"{metric_name_base.upper()}")
    ax.set_title(f"Per-Class {metric_name_base.upper()} across Models")
    ax.set_xticks(x_models)
    ax.set_xticklabels(short_labels)
    ax.legend(title="Class")
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"per_class_{metric_name_base}_across_models_bar.png"), dpi=200)
    plt.close(fig)

    # 折线图：每个类别一条线，X=模型
    fig, ax = plt.subplots(figsize=(9, 5))
    for idx, (col, label) in enumerate(class_metrics):
        values = summary_df[col].values
        ax.plot(x_models, values, marker='o', label=label, linewidth=2, markersize=6)
    ax.set_xlabel("Model")
    ax.set_ylabel(f"{metric_name_base.upper()}")
    ax.set_title(f"Per-Class {metric_name_base.upper()} across Models")
    ax.set_xticks(x_models)
    ax.set_xticklabels(short_labels)
    ax.legend(title="Class")
    ax.grid(True, linestyle='--', alpha=0.6)
    fig.tight_layout()
    fig.savefig(os.path.join(vis_dir, f"per_class_{metric_name_base}_across_models_line.png"), dpi=200)
    plt.close(fig)

print(f"所有可视化（柱状图+折线图）已保存至: {vis_dir}")